In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/dga-domain-detection-challenge/sample_submission.csv
/kaggle/input/competitions/dga-domain-detection-challenge/train.csv
/kaggle/input/competitions/dga-domain-detection-challenge/test.csv


In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv('/kaggle/input/competitions/dga-domain-detection-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/dga-domain-detection-challenge/test.csv')
sample = pd.read_csv('/kaggle/input/competitions/dga-domain-detection-challenge/sample_submission.csv')

print("TRAIN")
display(train.head())

print("TEST")
display(test.head())

print("SAMPLE")
display(sample.head())

print("Train columns:", train.columns.tolist())
print("Test columns:", test.columns.tolist())
print("Sample columns:", sample.columns.tolist())

TRAIN


,domain,label
0,c4wfahorcxq41ps0g0u,1
1,goievinskycattederifg,1
2,ypqbo2g854e4sb7re8i2e2e,1
3,cmikleasuredehydratorysagp.com,1
4,apodoc.saarland,0


TEST


,id,domain
0,0,direktorihosting.com
1,1,xbef.com
2,2,17929
3,3,shopcaliforniacustomroadsters.com
4,4,tujycagakykaqo


SAMPLE


,id,label
0,0,1
1,1,1
2,2,1
3,3,1
4,4,0


Train columns: ['domain', 'label']
Test columns: ['id', 'domain']
Sample columns: ['id', 'label']


In [3]:
import math
from collections import Counter

def entropy(s):
    s = str(s)
    if len(s) == 0:
        return 0
    
    counts = Counter(s)
    probs = [count / len(s) for count in counts.values()]
    
    return -sum(p * math.log2(p) for p in probs)


def get_sld(domain):
    domain = str(domain).lower()
    parts = domain.split('.')
    
    # если домен типа google.com, берем google
    # если домен уже без точки, оставляем как есть
    if len(parts) >= 2:
        return parts[-2]
    
    return parts[0]


def extract_features(df):
    df = df.copy()
    
    df['domain_str'] = df['domain'].astype(str).str.lower()
    df['sld'] = df['domain_str'].apply(get_sld)
    
    df['length'] = df['sld'].apply(len)
    df['num_digits'] = df['sld'].apply(lambda x: sum(ch.isdigit() for ch in x))
    df['num_letters'] = df['sld'].apply(lambda x: sum(ch.isalpha() for ch in x))
    df['num_vowels'] = df['sld'].apply(lambda x: sum(ch in 'aeiou' for ch in x))
    df['num_consonants'] = df['sld'].apply(lambda x: sum(ch.isalpha() and ch not in 'aeiou' for ch in x))
    df['num_hyphens'] = df['sld'].apply(lambda x: x.count('-'))
    df['unique_chars'] = df['sld'].apply(lambda x: len(set(x)))
    df['entropy'] = df['sld'].apply(entropy)
    
    df['digit_ratio'] = df['num_digits'] / df['length'].replace(0, 1)
    df['vowel_ratio'] = df['num_vowels'] / df['length'].replace(0, 1)
    df['consonant_ratio'] = df['num_consonants'] / df['length'].replace(0, 1)
    df['unique_ratio'] = df['unique_chars'] / df['length'].replace(0, 1)
    
    df['has_tld'] = df['domain_str'].apply(lambda x: int('.' in x))
    
    return df


train_feat = extract_features(train)
test_feat = extract_features(test)

print("Признаки созданы!")
display(train_feat.head())

Признаки созданы!


,domain,label,domain_str,sld,length,num_digits,num_letters,num_vowels,num_consonants,num_hyphens,unique_chars,entropy,digit_ratio,vowel_ratio,consonant_ratio,unique_ratio,has_tld
0,c4wfahorcxq41ps0g0u,1,c4wfahorcxq41ps0g0u,c4wfahorcxq41ps0g0u,19,5,14,3,11,0,16,3.932138,0.263158,0.157895,0.578947,0.842105,0
1,goievinskycattederifg,1,goievinskycattederifg,goievinskycattederifg,21,0,21,8,13,0,15,3.748995,0.000000,0.380952,0.619048,0.714286,0
2,ypqbo2g854e4sb7re8i2e2e,1,ypqbo2g854e4sb7re8i2e2e,ypqbo2g854e4sb7re8i2e2e,23,9,14,6,8,0,15,3.708132,0.391304,0.260870,0.347826,0.652174,0
3,cmikleasuredehydratorysagp.com,1,cmikleasuredehydratorysagp.com,cmikleasuredehydratorysagp,26,0,26,9,17,0,17,3.921030,0.000000,0.346154,0.653846,0.653846,1
4,apodoc.saarland,0,apodoc.saarland,apodoc,6,0,6,3,3,0,5,2.251629,0.000000,0.500000,0.500000,0.833333,1


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

feature_cols = [
    'length',
    'num_digits',
    'num_letters',
    'num_vowels',
    'num_consonants',
    'num_hyphens',
    'unique_chars',
    'entropy',
    'digit_ratio',
    'vowel_ratio',
    'consonant_ratio',
    'unique_ratio',
    'has_tld'
]

X = train_feat[feature_cols]
y = train_feat['label']

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight='balanced'
    )
)

model.fit(X_train, y_train)

val_proba = model.predict_proba(X_val)[:, 1]

print("Модель обучилась!")

Модель обучилась!


In [5]:
threshold = 0.7

val_pred = (val_proba >= threshold).astype(int)

print("Confusion matrix:")
print(confusion_matrix(y_val, val_pred))

print("\nClassification report:")
print(classification_report(y_val, val_pred))

print("ROC-AUC:", roc_auc_score(y_val, val_proba))

Confusion matrix:
[[1864835  102862]
 [ 725515  850746]]

Classification report:
              precision    recall  f1-score   support

           0       0.72      0.95      0.82   1967697
           1       0.89      0.54      0.67   1576261

    accuracy                           0.77   3543958
   macro avg       0.81      0.74      0.75   3543958
weighted avg       0.80      0.77      0.75   3543958

ROC-AUC: 0.8520241004387503


In [6]:
# Создаем предсказания для test

X_test = test_feat[feature_cols]

# Вероятность класса 1, то есть DGA
test_proba = model.predict_proba(X_test)[:, 1]

# Порог: чем выше, тем осторожнее модель ставит 1
threshold = 0.7
test_pred = (test_proba >= threshold).astype(int)

# Создаем submission НЕ через sample, потому что sample у тебя только на 11 строк
submission = pd.DataFrame({
    'id': test['id'].values,
    'label': test_pred
})

# Проверяем размеры
print("Размер test:", len(test))
print("Размер test_pred:", len(test_pred))
print("Размер submission:", len(submission))

# Сохраняем файл
submission.to_csv('submission.csv', index=False)

display(submission.head())
print("submission.csv saved!")

Размер test: 7594197
Размер test_pred: 7594197
Размер submission: 7594197


,id,label
0,0,0
1,1,0
2,2,0
3,3,1
4,4,0


submission.csv saved!
